In [1]:
import pandas as pd
import os

In [11]:
# read in the sample information: load the csv generated by pypette
df_sample = pd.read_csv("../../../config/samplesTestFast02_tiny.csv")

In [12]:
df_sample

,sample_name,sample_run,sample_path
0,connect_5k_pbmc_NGSC3_ch1_gex_1,run01,/idle/ric.cosr/ric.cosr/pedrini.edoardo/sample...
1,connect_5k_pbmc_NGSC3_ch1_gex_1,run01,/idle/ric.cosr/ric.cosr/pedrini.edoardo/sample...
2,connect_5k_pbmc_NGSC3_ch1_gex_2,run01,/idle/ric.cosr/ric.cosr/pedrini.edoardo/sample...
3,connect_5k_pbmc_NGSC3_ch1_gex_2,run01,/idle/ric.cosr/ric.cosr/pedrini.edoardo/sample...


In [13]:
# --- implementation for single runs ---

# 1. Sort the DataFrame by 'sample_name' and 'sample_run'
df_sample2 = df_sample.sort_values(by=['sample_name', 'sample_run']).reset_index(drop=True)

# 2. Create a cumulative count within each 'sample_name' group: cumcount() starts at 0, so add 1 to start numbering from 1
df_sample2['sample_instance_num'] = df_sample2.groupby('sample_name').cumcount() + 1

# 3. Create the new unique sample name string
df_sample2['sample_name_unique'] = df_sample2['sample_name'] + '_' + df_sample2['sample_instance_num'].astype(str)


In [14]:
df_sample2

,sample_name,sample_run,sample_path,sample_instance_num,sample_name_unique
0,connect_5k_pbmc_NGSC3_ch1_gex_1,run01,/idle/ric.cosr/ric.cosr/pedrini.edoardo/sample...,1,connect_5k_pbmc_NGSC3_ch1_gex_1_1
1,connect_5k_pbmc_NGSC3_ch1_gex_1,run01,/idle/ric.cosr/ric.cosr/pedrini.edoardo/sample...,2,connect_5k_pbmc_NGSC3_ch1_gex_1_2
2,connect_5k_pbmc_NGSC3_ch1_gex_2,run01,/idle/ric.cosr/ric.cosr/pedrini.edoardo/sample...,1,connect_5k_pbmc_NGSC3_ch1_gex_2_1
3,connect_5k_pbmc_NGSC3_ch1_gex_2,run01,/idle/ric.cosr/ric.cosr/pedrini.edoardo/sample...,2,connect_5k_pbmc_NGSC3_ch1_gex_2_2


In [15]:
# generate a pandas dataframe. Here it is expected only one path per sample/run.
aggregated_single_alt = df_sample2.groupby(['sample_name','sample_run']).agg(
    sample_run=('sample_run', 'unique'),  # Get unique 'sample_run' per group
    sample_path_string=('sample_path', lambda s: sorted(s.map(os.path.dirname).unique())),
    # needed to check for the existance of the actual input files
    sample_path_list=('sample_path','unique'),
    # needed to avoid issues with the *.mro files during processing
    sample_name_unique=('sample_name_unique', 'first')
)

# 3. Convert the aggregated DataFrame to the desired dictionary format
SAMPLES_single = aggregated_single_alt.to_dict('index')


In [16]:
SAMPLES_single

{('connect_5k_pbmc_NGSC3_ch1_gex_1',
  'run01'): {'sample_run': array(['run01'], dtype=object), 'sample_path_string': ['/idle/ric.cosr/ric.cosr/pedrini.edoardo/sample_data/sample_pbmcs/connect_5k_pbmc_NGSC3_ch1_fastqs_tiny/connect_5k_pbmc_NGSC3_ch1_gex_1_tiny'], 'sample_path_list': array(['/idle/ric.cosr/ric.cosr/pedrini.edoardo/sample_data/sample_pbmcs/connect_5k_pbmc_NGSC3_ch1_fastqs_tiny/connect_5k_pbmc_NGSC3_ch1_gex_1_tiny/connect_5k_pbmc_NGSC3_ch1_gex_1_S1_L004_R1_001.fastq.gz',
         '/idle/ric.cosr/ric.cosr/pedrini.edoardo/sample_data/sample_pbmcs/connect_5k_pbmc_NGSC3_ch1_fastqs_tiny/connect_5k_pbmc_NGSC3_ch1_gex_1_tiny/connect_5k_pbmc_NGSC3_ch1_gex_1_S1_L004_R2_001.fastq.gz'],
        dtype=object), 'sample_name_unique': 'connect_5k_pbmc_NGSC3_ch1_gex_1_1'},
 ('connect_5k_pbmc_NGSC3_ch1_gex_2',
  'run01'): {'sample_run': array(['run01'], dtype=object), 'sample_path_string': ['/idle/ric.cosr/ric.cosr/pedrini.edoardo/sample_data/sample_pbmcs/connect_5k_pbmc_NGSC3_ch1_fastqs_t

In [17]:
# --- implementation for merging runs ---

# generate a pandas dataframe. Here is expected a comma separated string of paths per sample.
aggregated_merge_alt = df_sample.groupby('sample_name').agg(
    # sample_run=('sample_run', lambda s: ",".join(s.unique())),
    # Aggregate sample_paths:
    # 1. Apply os.path.dirname to each path in the group (s.map(os.path.dirname))
    # 2. Find the unique directory names (.unique())
    # 3. Sort the unique directory names (sorted(...))
    # 4. Join the sorted unique directory names with a comma (",".join(...))
    sample_path_string=('sample_path', lambda s: ",".join(sorted(s.map(os.path.dirname).unique()))),
    # needed to check for the existance of the actual input files
    sample_path_list=('sample_path','unique')
)

# 3. Convert the aggregated DataFrame to the desired dictionary format
SAMPLES_merge = aggregated_merge_alt.to_dict('index')


In [18]:
SAMPLES_merge

{'connect_5k_pbmc_NGSC3_ch1_gex_1': {'sample_path_string': '/idle/ric.cosr/ric.cosr/pedrini.edoardo/sample_data/sample_pbmcs/connect_5k_pbmc_NGSC3_ch1_fastqs_tiny/connect_5k_pbmc_NGSC3_ch1_gex_1_tiny',
  'sample_path_list': array(['/idle/ric.cosr/ric.cosr/pedrini.edoardo/sample_data/sample_pbmcs/connect_5k_pbmc_NGSC3_ch1_fastqs_tiny/connect_5k_pbmc_NGSC3_ch1_gex_1_tiny/connect_5k_pbmc_NGSC3_ch1_gex_1_S1_L004_R1_001.fastq.gz',
         '/idle/ric.cosr/ric.cosr/pedrini.edoardo/sample_data/sample_pbmcs/connect_5k_pbmc_NGSC3_ch1_fastqs_tiny/connect_5k_pbmc_NGSC3_ch1_gex_1_tiny/connect_5k_pbmc_NGSC3_ch1_gex_1_S1_L004_R2_001.fastq.gz'],
        dtype=object)},
 'connect_5k_pbmc_NGSC3_ch1_gex_2': {'sample_path_string': '/idle/ric.cosr/ric.cosr/pedrini.edoardo/sample_data/sample_pbmcs/connect_5k_pbmc_NGSC3_ch1_fastqs_tiny/connect_5k_pbmc_NGSC3_ch1_gex_2_tiny',
  'sample_path_list': array(['/idle/ric.cosr/ric.cosr/pedrini.edoardo/sample_data/sample_pbmcs/connect_5k_pbmc_NGSC3_ch1_fastqs_tiny/co